In [15]:
import sys
!{sys.executable} -m pip install azure-storage-blob \
pandas \
numpy \
opencv-python-headless \
torch \
geopandas \
Pillow \
shapely \
transformers \
pyarrow \
opencv-python

  Using cached azure_storage_blob-12.28.0-py3-none-any.whl.metadata (26 kB)
  Using cached azure_core-1.39.0-py3-none-any.whl.metadata (48 kB)
  Using cached isodate-0.7.2-py3-none-any.whl.metadata (11 kB)
Using cached azure_storage_blob-12.28.0-py3-none-any.whl (431 kB)
Using cached azure_core-1.39.0-py3-none-any.whl (218 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 693.8 kB/s  0:00:07 eta 0:00:01
Using cached isodate-0.7.2-py3-none-any.whl (22 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [azure-storage-blob]zure-storage-blob]


In [5]:
!gdown 1ahs9K1ZHkqE-Dsbmh17RkU-zGONvRLzU

!mkdir output
!mkdir output/images


Downloading...
From (original): https://drive.google.com/uc?id=1ahs9K1ZHkqE-Dsbmh17RkU-zGONvRLzU
From (redirected): https://drive.google.com/uc?id=1ahs9K1ZHkqE-Dsbmh17RkU-zGONvRLzU&confirm=t&uuid=3c7c3232-fc6e-460b-b0fe-733ae53b0eba
To: /bounding_images.zip
100%|██████████████████████████████████████| 4.36G/4.36G [02:38<00:00, 27.4MB/s]
mkdir: cannot create directory ‘output’: File exists
mkdir: cannot create directory ‘output/images’: File exists


In [6]:
import zipfile
from tqdm import tqdm

zip_path = "bounding_images.zip"
out_dir = "output/images"

with zipfile.ZipFile(zip_path, 'r') as z:
    for file in z.namelist():
        z.extract(file, out_dir)

In [20]:
from huggingface_hub import login
login()


In [3]:
from azure.storage.blob import BlobServiceClient, ContainerClient, BlobBlock, BlobClient, StandardBlobTier, ContentSettings
# TODO: Replace <storage-account-name> with your actual storage account name
account_url = ""

blob_service_client = BlobServiceClient.from_connection_string(account_url)
container_name = "vastbuild"  # choose any name

container_client = blob_service_client.get_container_client(container_name)
# container_client.create_container()

In [9]:
import shutil

shutil.rmtree("/output/polygons")

In [4]:
import os
import re
import gc
import glob
import json
import time
import traceback
import cv2
import numpy as np
import pandas as pd
import torch
import geopandas as gpd

from PIL import Image
from concurrent.futures import ThreadPoolExecutor, Future
from collections import deque
from shapely.geometry import Polygon
from transformers import Sam3Processor, Sam3Model
import torch
import io
import os
import uuid

In [8]:
IMAGE_DIR       = "output/images"
OUT_DIR         = "output/polygons"
CHECKPOINT_FILE = "output/polygons/checkpoint.json"

BATCH_SIZE      = 24
FLUSH_EVERY     = 100
MIN_SCORE       = 0.18
MASK_THRESH     = 0.5
MIN_AREA_M2     = 10
IMG_SIZE        = 512
MIN_MASK_PX     = 20


# ──────────────────────────────────────────────
# Checkpoint
# ──────────────────────────────────────────────

class Checkpoint:

    def __init__(self, path: str):
        self.path      = path
        self.done: set = set()
        self.shard_id: int = 0
        self._load()

    def _load(self):
        if os.path.exists(self.path):
            try:
                with open(self.path) as f:
                    data = json.load(f)
                self.done     = set(data.get("done", []))
                self.shard_id = int(data.get("shard_id", 0))
                print(f"[checkpoint] Resumed — {len(self.done):,} tiles done, "
                      f"next shard: {self.shard_id}")
            except Exception as e:
                print(f"[checkpoint] Unreadable ({e}), starting fresh.")

    def seed(self, all_paths, resume_tile_id: int, resume_shard_id: int):
        """Reconstruct state when the checkpoint file was lost."""
        for p in all_paths:
            parsed = parse_filename(p)
            if parsed and parsed[0] <= resume_tile_id:
                self.done.add(parsed[0])
        self.shard_id = resume_shard_id
        print(f"[checkpoint] Seeded — {len(self.done):,} tiles marked done, "
              f"next shard: {self.shard_id}")
        self.save()

    def save(self):
        tmp = self.path + ".tmp"
        with open(tmp, "w") as f:
            json.dump({"done": list(self.done), "shard_id": self.shard_id}, f)
        os.replace(tmp, self.path)

    def mark_done(self, tile_ids: list):
        self.done.update(tile_ids)

    def is_done(self, tile_id: int) -> bool:
        return tile_id in self.done

    def next_shard(self) -> int:
        sid = self.shard_id
        self.shard_id += 1
        return sid


# ──────────────────────────────────────────────
# Filename helpers  — parsed once at startup (suggestion 9)
# ──────────────────────────────────────────────

_FNAME_RE = re.compile(
    r"tile_(\d+)_(-?[\d.]+)_(-?[\d.]+)_(-?[\d.]+)_(-?[\d.]+)\.jpg",
    re.IGNORECASE,
)

def parse_filename(path: str):
    m = _FNAME_RE.search(os.path.basename(path))
    if not m:
        return None
    return (
        int(m.group(1)),
        float(m.group(2)), float(m.group(3)),   # west, south
        float(m.group(4)), float(m.group(5)),   # east, north
    )


# ──────────────────────────────────────────────
# Polygon extraction  — OpenCV, no buffer(0) (suggestion 4)
#
# Why we removed poly.buffer(0):
#   OpenCV contours are topologically valid by construction — they trace
#   actual pixel boundaries without self-crossings.  Self-intersections
#   can appear after the float-precision geo projection, but only on
#   very thin 1-pixel-wide shapes that the MIN_MASK_PX gate already
#   removes.  Wrapping in try/except is cleaner than the expensive GEOS
#   repair call and preserves the same polygon count in practice.
# ──────────────────────────────────────────────

def mask_to_polygons(mask_u8, west, south, east, north, min_area_deg2=0.0):
    """
    mask_u8: uint8 numpy array (already thresholded on GPU — values 0 or 1).
    Returns a list of Shapely Polygons in geographic coordinates.
    """
    # Fast pixel-count gate — avoids contour work on near-empty masks
    if mask_u8.sum() < MIN_MASK_PX:
        return []

    h, w = mask_u8.shape
    lon_scale = (east  - west)  / w
    lat_scale = (north - south) / h

    contours, _ = cv2.findContours(mask_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    polys = []
    for cnt in contours:
        if len(cnt) < 3:
            continue

        # Vectorised pixel (col, row) → (lon, lat)
        pts  = cnt.squeeze(axis=1)
        lons = west  + pts[:, 0].astype(np.float64) * lon_scale
        lats = north - pts[:, 1].astype(np.float64) * lat_scale

        try:
            poly = Polygon(zip(lons, lats))
        except Exception:
            continue                       # degenerate contour — skip

        # Skip rather than repair: buffer(0) is an expensive GEOS call.
        # OpenCV output is almost always valid; truly invalid cases are
        # degenerate projections we don't want in the output anyway.
        if not poly.is_valid or poly.is_empty or poly.area < min_area_deg2:
            continue

        polys.append(poly)

    return polys


# ──────────────────────────────────────────────
# CPU worker  — polygon extraction in thread pool
# ──────────────────────────────────────────────

def _process_tile(tile_id, west, south, east, north,
                  masks_u8, scores, min_score, min_area_deg2):
    """
    masks_u8: list of uint8 numpy arrays (already thresholded).
    Returns column-wise (tile_ids, scores, geoms) to avoid per-polygon dict.
    """
    out_tile_ids = []
    out_scores   = []
    out_geoms    = []

    for mask_u8, score in zip(masks_u8, scores):
        if score < min_score:
            continue
        for poly in mask_to_polygons(mask_u8, west, south, east, north, min_area_deg2):
            out_tile_ids.append(tile_id)
            out_scores.append(float(score))
            out_geoms.append(poly)

    return out_tile_ids, out_scores, out_geoms


# ──────────────────────────────────────────────
# GPU helpers
# ──────────────────────────────────────────────

def flush_gpu():
    """Call ONLY on OOM — never per batch.
    empty_cache() forces a CUDA sync, blocking the GPU stream."""
    torch.cuda.empty_cache()
    gc.collect()

def gpu_mem_gb():
    return torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0


# ──────────────────────────────────────────────
# Model
# ──────────────────────────────────────────────

def load_model():
    processor = Sam3Processor.from_pretrained("facebook/sam3")
    model     = Sam3Model.from_pretrained("facebook/sam3")
    return processor, model


# ──────────────────────────────────────────────
# GPU inference
# ──────────────────────────────────────────────

def run_batch(processor, model, device, images):
    inputs = processor(
        images=images,
        text=["building"] * len(images),
        return_tensors="pt",
    ).to(device)

    # autocast: free 20-30% on A100/H100 for encoder-heavy models
    with torch.no_grad(), torch.cuda.amp.autocast(enabled=device.type == "cuda"):
        outputs = model(**inputs)

    results = processor.post_process_instance_segmentation(
        outputs,
        threshold=MIN_SCORE,
        mask_threshold=MASK_THRESH,
        target_sizes=inputs.get("original_sizes").tolist(),
    )

    # ── suggestion 7: threshold on GPU, transfer uint8 not float32 ──────
    # (m > MASK_THRESH) on GPU → bool tensor → uint8 numpy via .byte()
    # 4× less data over PCIe (1 byte vs 4 bytes per pixel).
    # mask_to_polygons receives uint8 directly; no numpy threshold step needed.
    cpu = []
    for r in results:
        cpu.append({
            "masks":  [(m > MASK_THRESH).byte().cpu().numpy() for m in r["masks"]],
            "scores": r["scores"].cpu().numpy(),
        })

    del outputs, inputs, results
    # No flush_gpu() — PyTorch caching allocator reuses VRAM automatically.
    return cpu


# ──────────────────────────────────────────────
# Parquet write  — runs in a dedicated background thread (suggestion 8)
# ──────────────────────────────────────────────

def _write_shard_to_disk(path, tile_ids, scores, geoms, ckpt, tile_ids_set,azure_save=False):
    """
    Executed in the write_pool thread.
    Writes the parquet file first, then updates the checkpoint.
    This ordering ensures we never mark a tile as done before its data is safe.
    """
    gdf = gpd.GeoDataFrame(
        {"tile_id": tile_ids, "score": scores},
        geometry=geoms,
        crs="EPSG:4326",
    )
    gdf.to_parquet(path, compression="snappy", index=False)
    if azure_save:
        upload_blob_file(path, blob_service_client, container_name)

    # Checkpoint update AFTER successful write
    ckpt.mark_done(list(tile_ids_set))
    ckpt.save()


def flush_shard_async(buf_tile_ids, buf_scores, buf_geoms,
                      ckpt, out_dir, write_pool, write_futures, azure_save=False):
    """Non-blocking shard flush. Returns immediately; write happens in background."""
    if not buf_geoms:
        return

    shard_id     = ckpt.next_shard()
    path         = os.path.join(out_dir, f"buildings_{shard_id:04d}.parquet")
    tile_ids_set = set(buf_tile_ids)

    # Take a snapshot of current buffers before clearing them in the caller
    fut = write_pool.submit(
        _write_shard_to_disk,
        path,
        list(buf_tile_ids),   # copy — caller will clear original
        list(buf_scores),
        list(buf_geoms),
        ckpt,
        tile_ids_set,
        azure_save
    )
    write_futures.append((fut, shard_id, len(buf_geoms), path))
    print(f"  ↑ Shard {shard_id:04d}  {len(buf_geoms):>5} polygons  → {path}  (writing…)")


def _drain_write_futures(write_futures, block=False):
    """Collect write futures that have completed, logging errors."""
    remaining = []
    for item in write_futures:
        fut, shard_id, n, path = item
        if block or fut.done():
            try:
                fut.result()
                print(f"  ✓ Shard {shard_id:04d}  {n:>5} polygons  saved  {path}")
            except Exception as e:
                print(f"  [WRITE ERROR] Shard {shard_id:04d}: {e}")
        else:
            remaining.append(item)
    write_futures[:] = remaining


# ──────────────────────────────────────────────
# Image loader  — used in prefetch thread pool
# ──────────────────────────────────────────────

def _load_image(path: str):
    try:
        return Image.open(path).convert("RGB")
    except Exception as e:
        print(f"  [WARN] {os.path.basename(path)}: {e}")
        return None


# ──────────────────────────────────────────────
# Main pipeline
# ──────────────────────────────────────────────

def run_inference(
    image_dir       = IMAGE_DIR,
    out_dir         = OUT_DIR,
    batch_size      = BATCH_SIZE,
    resume_tile_id  = None,
    resume_shard_id = None,
    cpu_workers     = None,
    io_workers      = 8,
    azure_save=False
):
    """
    resume_tile_id / resume_shard_id
    ---------------------------------
    Pass these when the checkpoint JSON was lost.
    Example — session ended at tile 4565, shard 67 was next:

        run_inference(resume_tile_id=4565, resume_shard_id=67)

    Checkpoint is re-created on first flush; subsequent runs need no args.

    Why ProcessPoolExecutor was NOT used for polygon extraction
    -----------------------------------------------------------
    Shapely calls GEOS (C library) which releases the GIL. OpenCV also
    releases the GIL. So ThreadPoolExecutor gives real parallelism on both.
    ProcessPool adds pickle roundtrip cost (numpy arrays + WKB-serialised
    Shapely objects) that typically exceeds the GIL savings for the task
    sizes here. Profile your specific hardware if you want to verify.
    """
    os.makedirs(out_dir, exist_ok=True)

    # cudnn autotuning: measures fastest kernel for fixed 512x512 inputs
    # on first batch, then reuses that choice for all subsequent batches.
    torch.backends.cudnn.benchmark = True

    all_paths = sorted(glob.glob(os.path.join(image_dir, "tile_*.jpg")))

    ckpt = Checkpoint(CHECKPOINT_FILE)
    if (not os.path.exists(CHECKPOINT_FILE) or len(ckpt.done) == 0) \
            and resume_tile_id is not None and resume_shard_id is not None:
        ckpt.seed(all_paths, resume_tile_id, resume_shard_id)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}  |  GPU: {gpu_mem_gb():.1f} GB used")

    processor, model = load_model()
    model = model.to(device).eval()

    MIN_AREA_DEG2 = MIN_AREA_M2 / (111_000 * 91_000)

    # ── suggestion 9: precompute parse_filename for every path once ──────
    # Previously called twice per path (filter step + prefetch step).
    # Now O(N) at startup, O(1) per access in the loop.
    path_meta = [
        (p, meta)
        for p in all_paths
        if (meta := parse_filename(p)) and not ckpt.is_done(meta[0])
    ]

    total = len(path_meta)
    print(f"Total on disk : {len(all_paths):,}")
    print(f"Already done  : {len(ckpt.done):,}")
    print(f"To process    : {total:,}\n")

    if not total:
        print("Nothing to do.")
        return

    n_cpu   = cpu_workers or os.cpu_count() or 4
    n_io    = io_workers
    n_total = max(n_cpu, n_io)

    buf_tile_ids: list = []
    buf_scores:   list = []
    buf_geoms:    list = []

    poly_total       = 0
    img_done         = 0
    imgs_since_flush = 0
    t0               = time.time()

    # Pending polygon-extraction futures
    poly_futures: deque[Future] = deque()

    # Pending parquet-write futures (list, drained lazily)
    write_futures: list = []

    def _drain_poly_ready():
        nonlocal poly_total
        while poly_futures and poly_futures[0].done():
            tids, scrs, gms = poly_futures.popleft().result()
            buf_tile_ids.extend(tids); buf_scores.extend(scrs); buf_geoms.extend(gms)
            poly_total += len(gms)

    def _drain_poly_all():
        nonlocal poly_total
        while poly_futures:
            tids, scrs, gms = poly_futures.popleft().result()
            buf_tile_ids.extend(tids); buf_scores.extend(scrs); buf_geoms.extend(gms)
            poly_total += len(gms)

    # Single shared executor handles both prefetch I/O and polygon threads.
    # Write pool is separate (1 worker) — serialises disk writes safely.
    with ThreadPoolExecutor(max_workers=n_total) as pool, \
         ThreadPoolExecutor(max_workers=1)      as write_pool:

        # Kick off first batch image load before the loop starts
        _next_slice = path_meta[:batch_size]
        _img_futs   = [pool.submit(_load_image, p) for p, _ in _next_slice]

        for batch_start in range(0, total, batch_size):
            current_slice = _next_slice

            # ── prefetch NEXT batch while we wait for current images ─────────
            _next_start = batch_start + batch_size
            _next_slice = path_meta[_next_start : _next_start + batch_size]
            _img_futs_next = [pool.submit(_load_image, p) for p, _ in _next_slice]

            # ── collect current batch images ──────────────────────────────────
            images, meta = [], []
            for img_f, (p, m) in zip(_img_futs, current_slice):
                img = img_f.result()
                if img is not None:
                    images.append(img)
                    meta.append(m)

            # Advance prefetch pointer
            _img_futs = _img_futs_next

            if not images:
                continue

            # ── GPU inference ─────────────────────────────────────────────────
            # OOM: retry with half-batch locally — does NOT shrink global batch_size.
            try:
                results = run_batch(processor, model, device, images)

            except torch.cuda.OutOfMemoryError:
                flush_gpu()
                time.sleep(2)
                retry_bs = max(1, len(images) // 2)
                print(f"  [OOM] retrying at bs={retry_bs}")

                results = []
                for chunk_s in range(0, len(images), retry_bs):
                    chunk_imgs = images[chunk_s : chunk_s + retry_bs]
                    chunk_meta = meta[chunk_s  : chunk_s + retry_bs]
                    try:
                        flush_gpu()
                        results.extend(run_batch(processor, model, device, chunk_imgs))
                    except torch.cuda.OutOfMemoryError:
                        flush_gpu()
                        time.sleep(3)
                        for m_ in chunk_meta:
                            print(f"  [OOM] Skipping tile {m_[0]}")
                            results.append({"masks": [], "scores": np.array([])})
                    except Exception as e:
                        for m_ in chunk_meta:
                            print(f"  [ERROR] tile {m_[0]}: {e}")
                            results.append({"masks": [], "scores": np.array([])})

            except Exception as e:
                print(f"  [ERROR] Batch {batch_start}: {e}")
                traceback.print_exc()
                flush_gpu()
                continue

            # ── submit CPU polygon extraction (non-blocking) ──────────────────
            # GPU is now free. Polygon futures run in thread pool while GPU
            # works on the next batch. masks_u8 are already uint8 from run_batch.
            for (tile_id, west, south, east, north), res in zip(meta, results):
                fut = pool.submit(
                    _process_tile,
                    tile_id, west, south, east, north,
                    res["masks"], res["scores"],
                    MIN_SCORE, MIN_AREA_DEG2,
                )
                poly_futures.append(fut)

            img_done         += len(meta)
            imgs_since_flush += len(meta)

            del images, results
            # No flush_gpu() per batch — caching allocator handles VRAM reuse.

            # Collect any polygon futures already done (overlap with GPU)
            _drain_poly_ready()
            # Drain completed write futures (non-blocking, just logging)
            _drain_write_futures(write_futures, block=False)

            # ── flush shard ───────────────────────────────────────────────────
            if imgs_since_flush >= FLUSH_EVERY:
                _drain_poly_all()             # wait for in-flight CPU polygon work
                flush_shard_async(            # non-blocking disk write
                    buf_tile_ids, buf_scores, buf_geoms,
                    ckpt, out_dir, write_pool, write_futures,azure_save
                )
                buf_tile_ids = []; buf_scores = []; buf_geoms = []
                imgs_since_flush = 0

            # ── progress ──────────────────────────────────────────────────────
            elapsed = time.time() - t0
            rate    = img_done / elapsed if elapsed else 1
            eta_m   = (total - img_done) / rate / 60
            print(
                f"  [{img_done:>6}/{total}]  "
                f"polys≈{poly_total:>7,}  "
                f"poly_q={len(poly_futures):>3}  "
                f"write_q={len(write_futures):>2}  "
                f"gpu={gpu_mem_gb():.1f}GB  "
                f"{rate:.1f}img/s  "
                f"ETA≈{eta_m:.0f}m"
            )

        # ── final drain ───────────────────────────────────────────────────────
        _drain_poly_all()
        flush_shard_async(
            buf_tile_ids, buf_scores, buf_geoms,
            ckpt, out_dir, write_pool, write_futures,
        )
        # Block until every write has landed on disk
        _drain_write_futures(write_futures, block=True)

    elapsed = time.time() - t0
    print(f"\nFinished. {img_done:,} images | {poly_total:,} polygons | {elapsed/60:.1f} min")


# ──────────────────────────────────────────────
# Merge shards
# ──────────────────────────────────────────────

def upload_blob_file(file_path, blob_service_client: BlobServiceClient, container_name: str):
    container_client = blob_service_client.get_container_client(container_name)

    blob_name = os.path.basename(file_path)

    blob_client = container_client.get_blob_client(blob_name)

    content_settings = ContentSettings(
        content_type="application/octet-stream"
    )

    # Upload in chunks with parallelism (important for parquet / large files)
    with open(file_path, "rb") as data:
        blob_client.upload_blob(
            data,
            overwrite=True,
            max_concurrency=8,          
            content_settings=content_settings,
        )

def load_model():
    processor = Sam3Processor.from_pretrained("facebook/sam3")
    model     = Sam3Model.from_pretrained("facebook/sam3", torch_dtype=torch.float16)
    return processor, model

def merge_shards(out_dir=OUT_DIR, out_file="buildings_final.parquet"):
    shards = sorted(glob.glob(os.path.join(out_dir, "buildings_*.parquet")))
    print(f"Merging {len(shards)} shards ...")
    gdf = gpd.GeoDataFrame(
        pd.concat([gpd.read_parquet(p) for p in shards], ignore_index=True),
        crs="EPSG:4326",
    )
    path = os.path.join(out_dir, out_file)
    gdf.to_parquet(path, compression="snappy", index=False)
    print(f"Done — {len(gdf):,} polygons → {path}")
    return gdf

In [ ]:
run_inference(
    image_dir       = "output/images",   
    out_dir         = "output/polygons",    
    azure_save=True
)

Device: cuda  |  GPU: 4.4 GB used


Loading weights:   0%|          | 0/1468 [00:00<?, ?it/s]

Total on disk : 57,810
Already done  : 0
To process    : 57,810

  [    24/57810]  polys≈      0  poly_q= 24  write_q= 0  gpu=6.1GB  3.0img/s  ETA≈324m
  [    48/57810]  polys≈    566  poly_q= 22  write_q= 0  gpu=6.1GB  3.0img/s  ETA≈325m
  [    72/57810]  polys≈    904  poly_q= 24  write_q= 0  gpu=6.1GB  3.0img/s  ETA≈325m
  [    96/57810]  polys≈  1,866  poly_q= 24  write_q= 0  gpu=6.1GB  2.9img/s  ETA≈328m
  ↑ Shard 0000   2036 polygons  → output/polygons/buildings_0000.parquet  (writing…)
  [   120/57810]  polys≈  2,036  poly_q=  0  write_q= 1  gpu=6.1GB  2.9img/s  ETA≈327m
  ✓ Shard 0000   2036 polygons  saved  output/polygons/buildings_0000.parquet
  [   144/57810]  polys≈  2,036  poly_q=  1  write_q= 0  gpu=6.1GB  2.9img/s  ETA≈327m
  [   168/57810]  polys≈  2,036  poly_q=  1  write_q= 0  gpu=6.1GB  2.9img/s  ETA≈330m
  [   192/57810]  polys≈  2,036  poly_q=  0  write_q= 0  gpu=6.1GB  2.9img/s  ETA≈332m
  [   216/57810]  polys≈  2,036  poly_q=  0  write_q= 0  gpu=6.1GB  2.9img/s

In [11]:
merge_shards()

Merging 438 shards ...
Done — 1,693,539 polygons → output/polygons/buildings_final.parquet


,tile_id,score,geometry
0,1,0.250000,"POLYGON ((-13.20272 27.15926, -13.20274 27.159..."
1,1,0.240112,"POLYGON ((-13.20047 27.159, -13.20049 27.15898..."
2,1,0.213989,"POLYGON ((-13.20282 27.15901, -13.20282 27.158..."
3,1,0.181152,"POLYGON ((-13.20149 27.15951, -13.2015 27.1595..."
4,1,0.325684,"POLYGON ((-13.20121 27.15889, -13.20123 27.158..."
...,...,...,...
1693534,57812,0.274902,"POLYGON ((-1.91655 34.74943, -1.91655 34.74942..."
1693535,57812,0.510254,"POLYGON ((-1.91592 34.75272, -1.91593 34.75271..."
1693536,57812,0.518555,"POLYGON ((-1.91555 34.75257, -1.91557 34.75255..."
1693537,57812,0.184082,"POLYGON ((-1.91724 34.75216, -1.91724 34.75214..."


In [17]:
final_polygons = gpd.read_parquet("output/polygons/buildings_final.parquet")

In [18]:
final_polygons.to_parquet("buildings_seg.parquet")

In [19]:
upload_blob_file("output/polygons/buildings_final.parquet",blob_service_client, container_name)

In [13]:
import json

with open(CHECKPOINT_FILE, "r") as file:
    checkpoint = json.load(file)

In [16]:
checkpoint['done'][-1]

57812